# Projet 1 : Préparation et Analyse de Données Financières

## Contexte

Ce projet a pour but de maîtriser la manipulation de données financières historiques en utilisant Python. Nous allons analyser trois géants de la tech : **Microsoft (MSFT)**, **Google (GOOGL)** et **Amazon (AMZN)** sur la période **2022–2024**.

## Objectifs

1. **Importation** — Récupérer les données boursières via `yfinance`
2. **Structuration** — Créer des DataFrames distincts par entreprise
3. **Nettoyage** — Vérifier et traiter les valeurs manquantes
4. **Exploration statistique** — Décrire les distributions de prix
5. **Visualisation** — Tracer l'évolution des prix et des volumes
6. **Corrélations** — Analyser les relations entre les trois actifs

---

> **Prérequis :** `pip install yfinance pandas matplotlib seaborn`

**Outils :** Python 3.12 · pandas · yfinance · matplotlib · seaborn

In [ ]:
# ── Paramètres globaux ──────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Paramètres de la période d'analyse
START = "2022-01-01"
END   = "2024-12-31"
TICKERS = ['MSFT', 'GOOGL', 'AMZN']
COLORS  = {'MSFT': '#4a9eff', 'GOOGL': '#f5a623', 'AMZN': '#00d084'}

# Style graphique
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print(f"Période d'analyse : {START} → {END}")
print(f"Actifs analysés   : {', '.join(TICKERS)}")

## 1. Importation des données

Téléchargement des données OHLCV (Open, High, Low, Close, Volume) depuis Yahoo Finance et création d'un DataFrame individuel par actif.

In [ ]:
# Téléchargement groupé puis séparation par actif
raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True)
raw.columns = raw.columns.droplevel('Ticker') if 'Ticker' in raw.columns.names else raw.columns

# Création de DataFrames individuels
MSFT  = raw.xs('MSFT',  axis=1, level='Ticker') if ('Close', 'MSFT') in raw.columns else raw
GOOGL = raw.xs('GOOGL', axis=1, level='Ticker') if ('Close', 'GOOGL') in raw.columns else raw
AMZN  = raw.xs('AMZN',  axis=1, level='Ticker') if ('Close', 'AMZN') in raw.columns else raw

# Approche alternative plus robuste
data = {}
for ticker in TICKERS:
    df = yf.download(ticker, start=START, end=END, auto_adjust=True, progress=False)
    df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
    data[ticker] = df

MSFT  = data['MSFT']
GOOGL = data['GOOGL']
AMZN  = data['AMZN']

print(f"MSFT  : {len(MSFT)} jours de trading")
print(f"GOOGL : {len(GOOGL)} jours de trading")
print(f"AMZN  : {len(AMZN)} jours de trading")
MSFT.head(5)

## 2. Nettoyage des données

Avant toute analyse, il est essentiel de vérifier la qualité des données : valeurs manquantes, doublons et cohérence des prix (High ≥ Low à tout moment).

In [ ]:
# ── Vérification des valeurs manquantes ─────────────────────────
print("=" * 45)
print("VALEURS MANQUANTES")
print("=" * 45)

for name, df in [('MSFT', MSFT), ('GOOGL', GOOGL), ('AMZN', AMZN)]:
    missing = df.isnull().sum()
    total   = missing.sum()
    status  = "✓ Aucune" if total == 0 else f"⚠ {total} valeurs manquantes"
    print(f"\n{name} — {status}")
    if total > 0:
        print(missing[missing > 0])

print("\n" + "=" * 45)
print("DOUBLONS")
print("=" * 45)
for name, df in [('MSFT', MSFT), ('GOOGL', GOOGL), ('AMZN', AMZN)]:
    dupes = df.index.duplicated().sum()
    print(f"{name} : {dupes} doublon(s)")

print("\n" + "=" * 45)
print("COHÉRENCE DES PRIX (High ≥ Low)")
print("=" * 45)
for name, df in [('MSFT', MSFT), ('GOOGL', GOOGL), ('AMZN', AMZN)]:
    anomalies = (df['High'] < df['Low']).sum()
    print(f"{name} : {anomalies} anomalie(s)")

## 3. Statistiques descriptives

Les statistiques descriptives permettent d'avoir une première vue sur la distribution des prix et des volumes de chaque actif.

In [ ]:
# Tableau comparatif des prix de clôture
stats_close = pd.DataFrame({
    'MSFT' : MSFT['Close'].describe(),
    'GOOGL': GOOGL['Close'].describe(),
    'AMZN' : AMZN['Close'].describe()
}).round(2)

print("Statistiques descriptives — Prix de clôture ($)")
print("=" * 50)
stats_close

## 4. Visualisation de l'évolution des prix

Traçons l'évolution des prix de clôture des trois actifs sur la même période pour faciliter la comparaison visuelle.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for ticker, df in [('MSFT', MSFT), ('GOOGL', GOOGL), ('AMZN', AMZN)]:
    ax.plot(df.index, df['Close'], label=ticker, color=COLORS[ticker], linewidth=1.8)

ax.set_title('Évolution des prix de clôture — MSFT, GOOGL, AMZN (2022–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Prix de clôture ($)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Performance normalisée (base 100)

Pour comparer équitablement des actifs avec des niveaux de prix différents, on normalise les cours en base 100 au premier jour de la période.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for ticker, df in [('MSFT', MSFT), ('GOOGL', GOOGL), ('AMZN', AMZN)]:
    normalized = (df['Close'] / df['Close'].iloc[0]) * 100
    ax.plot(df.index, normalized, label=ticker, color=COLORS[ticker], linewidth=1.8)

ax.axhline(y=100, color='white', linestyle='--', linewidth=0.8, alpha=0.5, label='Base 100')
ax.set_title('Performance normalisée (base 100) — MSFT, GOOGL, AMZN (2022–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Performance (base 100)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Performance totale sur la période
print("Performance totale sur la période :")
for ticker, df in [('MSFT', MSFT), ('GOOGL', GOOGL), ('AMZN', AMZN)]:
    perf = ((df['Close'].iloc[-1] / df['Close'].iloc[0]) - 1) * 100
    print(f"  {ticker} : {perf:+.1f}%")

## 6. Analyse des volumes

Le volume reflète l'activité du marché. Un volume élevé confirme la force d'un mouvement de prix.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for ax, (ticker, df) in zip(axes, [('MSFT', MSFT), ('GOOGL', GOOGL), ('AMZN', AMZN)]):
    ax.bar(df.index, df['Volume'] / 1e6, color=COLORS[ticker], alpha=0.7, width=1)
    ax.set_ylabel('Volume (M)')
    ax.set_title(ticker)
    mean_vol = df['Volume'].mean() / 1e6
    ax.axhline(y=mean_vol, color='white', linestyle='--', linewidth=0.8, alpha=0.6, label=f'Moyenne : {mean_vol:.0f}M')
    ax.legend(fontsize=9)

fig.suptitle('Volume journalier — MSFT, GOOGL, AMZN (2022–2024)', fontsize=13, fontweight='bold')
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Analyse des corrélations

La corrélation entre actifs est fondamentale en gestion de portefeuille. Des actifs fortement corrélés offrent peu de diversification.

On calcule les corrélations sur les **rendements journaliers** (et non les prix) pour éviter les effets de tendance.

In [ ]:
# Calcul des rendements journaliers
returns = pd.DataFrame({
    'MSFT' : MSFT['Close'].pct_change().dropna(),
    'GOOGL': GOOGL['Close'].pct_change().dropna(),
    'AMZN' : AMZN['Close'].pct_change().dropna()
})

# Matrice de corrélation
corr_matrix = returns.corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = pd.DataFrame(False, index=corr_matrix.index, columns=corr_matrix.columns)
import numpy as np
mask_upper = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=0, vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 13, 'weight': 'bold'}
)
ax.set_title('Matrice de corrélation des rendements journaliers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nInterprétation :")
pairs = [('MSFT','GOOGL'), ('MSFT','AMZN'), ('GOOGL','AMZN')]
for a, b in pairs:
    r = corr_matrix.loc[a, b]
    niveau = 'forte' if r > 0.75 else 'modérée' if r > 0.5 else 'faible'
    print(f"  {a} / {b} : {r:.2f} → corrélation {niveau}")

## Conclusion

Ce projet a posé les bases de l'analyse de données financières avec Python :

- Les données MSFT, GOOGL et AMZN sont **propres** (aucune valeur manquante, aucun doublon)
- Les trois actifs présentent des **niveaux de prix très différents**, d'où l'importance de la normalisation en base 100 pour comparer leurs performances
- Les rendements sont **fortement corrélés** entre les trois valeurs tech, ce qui est cohérent avec leur appartenance au même secteur — cela limitera les bénéfices de diversification entre ces actifs

**Prochaine étape →** Projet 2 : Analyse quantitative du risque de marché